# PySpark Transformations - Learning Journey

This notebook demonstrates fundamental PySpark transformations including:
- DataFrame creation with different schema approaches
- Data selection and filtering
- Type casting and column operations
- Writing data to CSV

Working with a realistic employee dataset containing duplicates and missing values.

In [ ]:
# Initialize Spark Session
# findspark helps Python find Spark installation
import findspark

findspark.init()

from pyspark.sql import SparkSession

# Create Spark session with app name and local mode using all available cores
spark = SparkSession.builder.appName("Transformations").master("local[*]").getOrCreate()

In [ ]:
# Define schema using DDL string format (all columns as STRING type)
# This approach is simple but requires manual type casting later
emp_schema = """
    emp_id STRING, 
    dept_id STRING, 
    name STRING, 
    age STRING, 
    gender STRING, 
    hire_date STRING,
    salary STRING
"""

# Sample employee data (20 rows)
# Intentionally includes real-world data issues:
# - Duplicate records (rows 1 & 4, rows 2 & 13)
# - Missing values (empty strings)
# - Inconsistent gender formats (Male/M, Female/F)
# - Null values represented as string "null"
emp_data = [
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],
    ["003", "101", "Priya S", "24", "Female", "2024-03-10", "62000"],
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],  # duplicate
    ["004", "103", "Meena R", "", "Female", "2022-11-20", ""],  # missing age & salary
    ["005", "102", "Saravanan", "35", "Male", "", "95000"],
    ["006", "104", "Karthik P", "29", "Male", "2024-09-05", "72000"],
    ["007", "", "Deepika Menon", "26", "Female", "2023-02-28", "68000"],  # missing dept
    ["008", "101", "Mohan Raj", "42", "Male", "2020-05-12", "120000"],
    ["009", "103", "Anjali", "23", "F", "2024-07-19", "58000"],
    ["010", "102", "Ramesh Kumar", "31", "M", "2021-10-01", "85000"],
    ["011", "105", "Swathi", "", "Female", "2025-02-10", "52000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],  # duplicate
    ["012", "101", "Vikram Singh", "38", "Male", "2019-08-25", "105000"],
    ["013", "104", "Preethi K", "27", "Female", "", "64000"],
    ["014", "103", "Naveen", "", "Male", "2024-01-15", "null"],  # explicit null
    ["015", "102", "Lavanya", "24", "Female", "2024-04-30", "61000"],
    ["016", "", "Sundar", "45", "M", "2018-03-05", "92000"],
    ["017", "101", "Kavya Sri", "22", "F", "2025-03-01", "48000"],
    ["018", "106", "Abdul Rahman", "33", "Male", "2022-12-12", "88000"],
]

In [ ]:
# Create DataFrame using DDL string schema
# All columns are STRING type at this point
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [ ]:
# Display the schema - notice all columns are string type
emp.printSchema()

In [ ]:
# Same employee data but with proper integer types for age column
# This demonstrates the advantage of defining correct types from the start
data_spark = [
    ["001", "101", "Vishnu", 21, "Male", "2025-01-01", "45000"],
    ["002", "102", "Arun Kumar", 28, "Male", "2023-06-15", "78000"],
    ["003", "101", "Priya S", 24, "Female", "2024-03-10", "62000"],
    ["001", "101", "Vishnu", 21, "Male", "2025-01-01", "45000"],  # duplicate
    ["004", "103", "Meena R", 20, "Female", "2022-11-20", ""],  # missing salary
    ["005", "102", "Saravanan", 35, "Male", "", "95000"],
    ["006", "104", "Karthik P", 29, "Male", "2024-09-05", "72000"],
    ["007", "", "Deepika Menon", 26, "Female", "2023-02-28", "68000"],  # missing dept
    ["008", "101", "Mohan Raj", 42, "Male", "2020-05-12", "120000"],
    ["009", "103", "Anjali", 23, "F", "2024-07-19", "58000"],
    ["010", "102", "Ramesh Kumar", 31, "M", "2021-10-01", "85000"],
    ["011", "105", "Swathi", 22, "Female", "2025-02-10", "52000"],
    ["002", "102", "Arun Kumar", 28, "Male", "2023-06-15", "78000"],  # duplicate
    ["012", "101", "Vikram Singh", 38, "Male", "2019-08-25", "105000"],
    ["013", "104", "Preethi K", 27, "Female", "", "64000"],
    ["014", "103", "Naveen", 33, "Male", "2024-01-15", "null"],  # explicit null
    ["015", "102", "Lavanya", 24, "Female", "2024-04-30", "61000"],
    ["016", "", "Sundar", 45, "M", "2018-03-05", "92000"],
    ["017", "101", "Kavya Sri", 22, "F", "2025-03-01", "48000"],
    ["018", "106", "Abdul Rahman", 33, "Male", "2022-12-12", "88000"],
]

In [ ]:
# Define schema using StructType (programmatic approach)
# This method provides:
# - Explicit type definitions
# - Better performance (no implicit conversions)
# - Type safety and validation

from pyspark.sql.types import IntegerType, StringType, StructField, StructType

schema_spark = StructType(
    [
        StructField("emp_id", StringType(), True),  # True = nullable
        StructField("dept_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),  # Age as Integer
        StructField("gender", StringType(), True),
        StructField("hire_date", StringType(), True),
        StructField("salary", StringType(), True),
    ]
)

In [ ]:
# Create DataFrame with StructType schema
# Age column is properly typed as Integer from the start
emp_final = spark.createDataFrame(data=data_spark, schema=schema_spark)

In [ ]:
# Verify schema - age should now be IntegerType
emp_final.printSchema()

In [ ]:
# Access the schema object directly
# Useful for programmatic schema inspection
emp_final.schema

In [ ]:
# Display the DataFrame contents
# Shows all 20 rows including duplicates and missing values
emp_final.show()

In [ ]:
# TRANSFORMATION: Filter employees with salary > 50000
# Note: Since salary is STRING type, comparison works lexicographically
# This is a lazy transformation - no execution until an action is called
emp_sal = emp_final.where("salary > 50000")

In [ ]:
# ACTION: Display filtered results
# This triggers execution of the transformation
emp_sal.show()

In [ ]:
# Write filtered data to CSV format
# Creates a directory with partitioned CSV files
emp_sal.write.format("csv").save(
    "/home/vishnu/pyspark-jupyter/codes/data/load_outputs/emp_salary.csv"
)

In [ ]:
# Display original DataFrame for comparison
emp.show()

In [ ]:
# TRANSFORMATION: Select specific columns using different methods
# Demonstrates three ways to reference columns:
# 1. col() function - standard approach
# 2. expr() function - for SQL expressions
# 3. DataFrame.column - dot notation

from pyspark.sql.functions import col, expr

emp_filtered = emp.select(col("emp_id"), expr("dept_id"), emp.name, emp.age)

In [ ]:
# Display selected columns
emp_filtered.show()

In [ ]:
# Check the schema of filtered DataFrame
emp.printSchema()

In [ ]:
# TRANSFORMATION: Type casting and column renaming using expr()
# - Rename emp_id to employee_id using alias
# - Cast age from STRING to INT
# - Keep name as-is
emp_casted = emp_filtered.select(
    expr("emp_id as employee_id"), "name", expr("cast(age as int) as age")
)

In [ ]:
# Verify that age is now IntegerType
emp_casted.printSchema()

In [ ]:
# ALTERNATIVE METHOD: Using selectExpr() for the same transformation
# selectExpr() allows SQL expressions directly without expr() wrapper
# More concise when working with multiple SQL expressions
emp_casted_1 = emp_filtered.selectExpr(
    "emp_id as employee_id", "name", "cast(age as int) as age"
)

In [ ]:
# Verify schema - should be identical to emp_casted
emp_casted_1.printSchema()

In [ ]:
# CHAINED TRANSFORMATIONS: Select columns and filter in one statement
# 1. Select employee_id, name, age
# 2. Filter for employees older than 30
# Since age is now INTEGER, numeric comparison works correctly
emp_final = emp_casted.select("employee_id", "name", "age").where("age>30")

In [ ]:
# Display final result - employees over 30 years old
emp_final.show()

In [ ]:
# BONUS: Parse DDL string schema into StructType
# This internal function converts DDL string format to StructType object
# Useful for understanding how Spark processes string schemas

schema_str = "name string, age int"

from pyspark.sql.types import _parse_datatype_string

schema_spark = _parse_datatype_string(schema_str)

In [ ]:
# Display the parsed schema structure
schema_spark